In [114]:
import os
import time
import random
import numpy as np
import mlflow 
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error


if torch.backends.mps.is_available():
    device = torch.device("mps")      # Mac GPU (Apple Silicon)
elif torch.cuda.is_available():
    device = torch.device("cuda")     # Nvidia GPU
else:
    device = torch.device("cpu")

# device = torch.device("cpu")  # Force CPU for testing time
random_seed = 42


mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("z_prediction")
mlflow.enable_system_metrics_logging()


## Use Hugos functions

In [107]:
def split_csvfiles(datafolder, random_seed):
    csv_files = []
    for f in os.listdir(datafolder):
        if f.endswith(".csv"):
            csv_files.append(f)
    
    random.seed(random_seed)
    random.shuffle(csv_files)

    # Train/Validation/Test with proportions 80/10/10
    train_n = int(len(csv_files) * 0.8)
    val_n = int(len(csv_files) * 0.1)
    test_n = len(csv_files) - train_n - val_n

    # Split
    train_files = csv_files[:train_n]
    val_files = csv_files[train_n: train_n + val_n]
    test_files = csv_files[train_n + test_n:]

    return train_files, val_files, test_files

def split_data_timeseries(datafolder):
    csv_files = []
    for f in os.listdir(datafolder):
        if f.endswith(".csv"):
            csv_files.append(os.path.join(datafolder, f)) 

    max_len = 0
    for f in csv_files:      
        max_len = max(max_len, len(pd.read_csv(f)))
    
    all_X = []
    all_y = []

    for file in csv_files:
        df      = pd.read_csv(file)
        X, y    = input_target_split(df)

        X = torch.tensor(X.values, dtype=torch.float32)
        y = torch.tensor(y.values, dtype=torch.float32)

        pad_len = max_len - X.shape[0]
        if pad_len > 0:
            X = torch.cat([X, torch.zeros(pad_len, X.shape[1])], dim=0)
            y = torch.cat([y, torch.zeros(pad_len, y.shape[1])], dim=0)

        all_X.append(X)
        all_y.append(y)

    all_X = torch.stack(all_X)  # (n_files, max_len, 26)
    all_y = torch.stack(all_y)  # (n_files, max_len, 13)


    return all_X, all_y


def load(files, data_dir):
    dataframes = []

    for f in files:
        path = os.path.join(data_dir, f)   
        df = pd.read_csv(path)           
        dataframes.append(df)             

    combined = pd.concat(dataframes, ignore_index=True)  # combine all

    return combined


def input_target_split(dataframe):
    input_cols = []
    target_cols = []
    for c in dataframe.columns:
        if c.endswith("_x") or c.endswith("_y"):
            input_cols.append(c)
        if c.endswith("_z"):
            target_cols.append(c)
    
    input_data = dataframe[input_cols]
    target_data = dataframe[target_cols]
    return input_data, target_data

## Load in the data

In [108]:
datafolder = "../../MainProject/Assignment9/data/kinect_good_preprocessed"
random_seed = 42

train_files, val_files, test_files = split_csvfiles(datafolder, random_seed)

train_data = load(train_files, datafolder)
val_data = load(val_files, datafolder)
test_data = load(test_files, datafolder)

x_train, y_train = input_target_split(train_data)
x_val, y_val = input_target_split(val_data)
x_test, y_test = input_target_split(test_data)


x_train = torch.tensor(x_train.values, dtype=torch.float32).to(device)
y_train = torch.tensor(y_train.values, dtype=torch.float32).to(device)
x_val = torch.tensor(x_val.values, dtype=torch.float32).to(device)
y_val = torch.tensor(y_val.values, dtype=torch.float32).to(device)
x_test = torch.tensor(x_test.values, dtype=torch.float32).to(device)
y_test = torch.tensor(y_test.values, dtype=torch.float32).to(device)

# Smaller dataset to "force" overfitting
"""
x_train = x_train[:50]
y_train = y_train[:50]
"""

'\nx_train = x_train[:50]\ny_train = y_train[:50]\n'

## Train the model

Hyperparameters and what we do with them: 

* "epochs"

* "optimizer": (SGD, RMSprop, Adam, …)

* loss functition: MSE, MAE 

* model_type

* Hidden layers: One or several hidden layers with varying  number of nodes, Variants of layer types, Activation and Initialization


* "activation": Video said ReLU was good choice so this is not a function that we are super intrested in. maxout also option but slower??



In [ ]:
def init_weights(m):
        if isinstance(m, nn.Linear):
            #nn.init.xavier_uniform_(m.weight)  # good for Tanh
            nn.init.kaiming_uniform_(m.weight) # good for ReLU
            #nn.init.zeros_(m.bias)

    
class ZPredictor(nn.Module):
    def __init__(self, hidden_layers: list, activation=nn.ReLU(), layer_type = "Dense", dropout=0.0):
        super().__init__()
        
        layers = []
        input_size = 26  # 13 joints x 2 (x, y)
        
        if layer_type == "Dense":
            for hidden_size in hidden_layers:
                layers.append(nn.Linear(input_size, hidden_size))
                layers.append(activation)
                if dropout > 0:
                    layers.append(nn.Dropout(dropout))
                input_size = hidden_size
            
            layers.append(nn.Linear(input_size, 13))  # 13 joints z output

        
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)
    





## Code for reccurant neural networks

* In our case, lstm or gru

In [110]:
class zPredictor_timeseries(nn.Module):
    def __init__(self, hidden_layers: list, layer_type = "LSTM", dropout=0.0):
        super().__init__()
        self.layer_type = layer_type
        
        layers = []
        input_size = 26  # 13 joints x 2 (x, y)
        

        rnn_class = nn.LSTM if layer_type == "LSTM" else nn.GRU
        self.rnns  = nn.ModuleList()
        self.drops = nn.ModuleList()

        sizes = [input_size] + hidden_layers
        for i in range(len(hidden_layers)):
            self.rnns.append(rnn_class(sizes[i], sizes[i+1], batch_first=True))
            self.drops.append(nn.Dropout(dropout) if dropout > 0 else nn.Identity())

        self.fc_out = nn.Linear(hidden_layers[-1], 13)

        
        self.network = nn.Sequential(*layers)
    
    def forward(self, x):
        if self.layer_type == "Dense":
            return self.network(x)

        elif self.layer_type in ("LSTM", "GRU"):
            for rnn, drop in zip(self.rnns, self.drops):
                x, _ = rnn(x)   # unpack tuple
                x = drop(x)
            return self.fc_out(x)  # (batch, max_len, 13)

# Train the non time series network

In [111]:
params = {
    "hidden_layers": [256, 128, 128, 64, 64],
    "activation": nn.ReLU(),
    "layer_type": "Dense",
    "dropout": 0,
    "learning_rate": 0.0005,
    "batch_size": 64,
    "epochs": 100,
    "run_name": "test_speed_gpu",
    #"patience": 5
}

model = ZPredictor(hidden_layers=params["hidden_layers"], activation=params["activation"], 
                   layer_type=params["layer_type"], dropout=params["dropout"]).to(device)
model.apply(init_weights) 

loss_fn = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=params["learning_rate"])

In [112]:
with mlflow.start_run(run_name=params["run_name"]) as run:
    mlflow.log_params(params)

    # Create DataLoaders for batching
    train_dataset = TensorDataset(x_train, y_train)
    val_dataset = TensorDataset(x_val, y_val)
    test_dataset = TensorDataset(x_test, y_test)
    
    train_loader = DataLoader(train_dataset, batch_size=params["batch_size"], shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=params["batch_size"], shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=params["batch_size"], shuffle=False)

    best_val_loss = float("inf")
    epochs_no_improve = 0
    
    for epoch in range(params["epochs"]):

        # Train step with batching
        model.train()
        train_losses = []
        train_maes = []
        train_biases = []
        
        for batch_x, batch_y in train_loader:
            optimizer.zero_grad()
            predictions = model(batch_x)
            loss = loss_fn(predictions, batch_y)
            loss.backward()
            optimizer.step()
            
            # Collect batch metrics
            train_losses.append(loss.item())
            train_maes.append(torch.mean(torch.abs(predictions - batch_y)).item())
            train_biases.append(torch.mean(predictions - batch_y).item())
        
        # Average training metrics
        avg_train_loss = np.mean(train_losses)
        avg_train_mae = np.mean(train_maes)
        avg_train_bias = np.mean(train_biases)

        # Evaluation step with batching
        model.eval()
        val_losses = []
        val_maes = []
        val_biases = []
        all_val_preds = []
        all_val_true = []
        
        with torch.no_grad():
            for batch_x, batch_y in val_loader:
                val_predictions = model(batch_x)
                loss = loss_fn(val_predictions, batch_y)
                
                val_losses.append(loss.item())
                val_maes.append(torch.mean(torch.abs(val_predictions - batch_y)).item())
                val_biases.append(torch.mean(val_predictions - batch_y).item())
                
                # Store for R² calculation
                all_val_preds.append(val_predictions.cpu().numpy().flatten())
                all_val_true.append(batch_y.cpu().numpy().flatten())
        
        # Average validation metrics
        avg_val_loss = np.mean(val_losses)
        avg_val_mae = np.mean(val_maes)
        avg_val_bias = np.mean(val_biases)
        
        # Calculate R² on all validation data
        val_preds_flat = np.concatenate(all_val_preds)
        val_true_flat = np.concatenate(all_val_true)
        val_r2 = r2_score(val_true_flat, val_preds_flat)
        
        # Calculate training R² (optional - using full training data)
        with torch.no_grad():
            train_predictions_full = model(x_train)
            train_r2 = r2_score(
                y_train.cpu().numpy().flatten(), 
                train_predictions_full.cpu().numpy().flatten()
            )

        # Log per epoch metrics with step
        mlflow.log_metrics({
            "train_loss": avg_train_loss,
            "train_mae": avg_train_mae,
            "train_r2": train_r2,
            "train_bias": avg_train_bias,
            "val_loss": avg_val_loss,
            "val_mae": avg_val_mae,
            "val_r2": val_r2,
            "val_bias": avg_val_bias
        }, step=epoch)

        print(f"Epoch {epoch+1}/{params['epochs']} "
              f"train_loss: {avg_train_loss:.4f} "
              f"train_mae: {avg_train_mae:.4f} "
              f"val_loss: {avg_val_loss:.4f} "
              f"val_mae: {avg_val_mae:.4f} cm")

        """
        # Early stopping
        if params.get("patience"):
            if avg_val_loss < best_val_loss:
                best_val_loss = avg_val_loss
                epochs_no_improve = 0
                mlflow.pytorch.log_model(model, artifact_path="best_model")
            else:
                epochs_no_improve += 1

            if epochs_no_improve >= params["patience"]:
                print(f"Early stopping at epoch {epoch+1}")
                break
        """

    # Test evaluation with batching
    model.eval()
    test_losses = []
    test_maes = []
    test_biases = []
    all_test_preds = []
    all_test_true = []
    
    with torch.no_grad():
        for batch_x, batch_y in test_loader:
            test_predictions = model(batch_x)
            loss = loss_fn(test_predictions, batch_y)
            
            test_losses.append(loss.item())
            test_maes.append(torch.mean(torch.abs(test_predictions - batch_y)).item())
            test_biases.append(torch.mean(test_predictions - batch_y).item())
            
            all_test_preds.append(test_predictions.cpu().numpy().flatten())
            all_test_true.append(batch_y.cpu().numpy().flatten())
    
    # Average test metrics
    avg_test_loss = np.mean(test_losses)
    avg_test_mae = np.mean(test_maes)
    avg_test_bias = np.mean(test_biases)
    
    # Calculate test R²
    test_preds_flat = np.concatenate(all_test_preds)
    test_true_flat = np.concatenate(all_test_true)
    test_r2 = r2_score(test_true_flat, test_preds_flat)

    mlflow.log_metrics({
        "test_loss": avg_test_loss,
        "test_mae": avg_test_mae,
        "test_r2": test_r2,
        "test_bias": avg_test_bias,  
    })

    mlflow.pytorch.log_model(model, artifact_path=f"final_model_{params['run_name']}")
    
    print(f"\n✅ Test Results - Loss: {avg_test_loss:.4f}, MAE: {avg_test_mae:.4f}, R²: {test_r2:.4f}")

2026/04/13 17:10:08 INFO mlflow.system_metrics.system_metrics_monitor: Skip logging GPU metrics. Set logger level to DEBUG for more details.
2026/04/13 17:10:08 INFO mlflow.system_metrics.system_metrics_monitor: Started monitoring system metrics.
2026/04/13 17:10:10 INFO mlflow.system_metrics.system_metrics_monitor: Stopping system metrics monitoring...
2026/04/13 17:10:10 INFO mlflow.system_metrics.system_metrics_monitor: Successfully terminated system metrics monitoring!


🏃 View run test_speed_gpu at: http://127.0.0.1:5000/#/experiments/1/runs/fed2c4fdbafe40e7960d81089140df76
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


KeyboardInterrupt: 

# Train the timeseries network

In [116]:
params = {
    "hidden_layers": [256, 128, 128, 64, 64],
    "layer_type": "GRU",
    "dropout": 0.2,
    "learning_rate": 0.0005,
    "batch_size": 64,
    "epochs": 1000,
    "run_name": "first_timeseries_with_gru",
    #"patience": 5
}

model = zPredictor_timeseries(hidden_layers=params["hidden_layers"], layer_type=params["layer_type"], dropout=params["dropout"]).to(device)

loss_fn = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=params["learning_rate"])

datafolder = "../../MainProject/Assignment9/data/kinect_good_preprocessed"
X, Y = split_data_timeseries(datafolder)

train_x, test_x, train_y, test_y = train_test_split(X, Y, test_size=0.1, random_state=random_seed)

# then split remaining 90% into train (80%) and val (10%)
train_x, val_x, train_y, val_y = train_test_split(train_x, train_y, test_size=0.111, random_state=random_seed)
# 0.111 of 90% ≈ 10% of total



print(f"Train: {len(train_x)}, Val: {len(val_x)}, Test: {len(test_x)}")


train_x, train_y = train_x.to(device), train_y.to(device)
val_x, val_y = val_x.to(device), val_y.to(device)
test_x, test_y = test_x.to(device), test_y.to(device)



Train: 143, Val: 18, Test: 18


In [117]:
with mlflow.start_run(run_name=params["run_name"]) as run:
    mlflow.log_params(params)

    train_dataset = TensorDataset(train_x, train_y)
    val_dataset = TensorDataset(val_x, val_y)
    test_dataset = TensorDataset(test_x, test_y)
    
    train_loader = DataLoader(train_dataset, batch_size=params["batch_size"], shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=params["batch_size"], shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=params["batch_size"], shuffle=False)

    best_val_loss = float("inf")
    epochs_no_improve = 0
    
    for epoch in range(params["epochs"]):

        # Train step with batching
        model.train()
        train_losses = []
        train_maes = []
        train_biases = []
        
        for batch_x, batch_y in train_loader:
            optimizer.zero_grad()
            predictions = model(batch_x)
            loss = loss_fn(predictions, batch_y)
            loss.backward()
            optimizer.step()
            
            # Collect batch metrics
            train_losses.append(loss.item())
            train_maes.append(torch.mean(torch.abs(predictions - batch_y)).item())
            train_biases.append(torch.mean(predictions - batch_y).item())
        
        # Average training metrics
        avg_train_loss = np.mean(train_losses)
        avg_train_mae = np.mean(train_maes)
        avg_train_bias = np.mean(train_biases)

        # Evaluation step with batching
        model.eval()
        val_losses = []
        val_maes = []
        val_biases = []
        all_val_preds = []
        all_val_true = []
        
        with torch.no_grad():
            for batch_x, batch_y in val_loader:
                val_predictions = model(batch_x)
                loss = loss_fn(val_predictions, batch_y)
                
                val_losses.append(loss.item())
                val_maes.append(torch.mean(torch.abs(val_predictions - batch_y)).item())
                val_biases.append(torch.mean(val_predictions - batch_y).item())
                
                # Store for R² calculation
                all_val_preds.append(val_predictions.cpu().numpy().flatten())
                all_val_true.append(batch_y.cpu().numpy().flatten())
        
        # Average validation metrics
        avg_val_loss = np.mean(val_losses)
        avg_val_mae = np.mean(val_maes)
        avg_val_bias = np.mean(val_biases)
        
        # Calculate R² on all validation data
        val_preds_flat = np.concatenate(all_val_preds)
        val_true_flat = np.concatenate(all_val_true)
        val_r2 = r2_score(val_true_flat, val_preds_flat)
        
        # Calculate training R² (using full training data)
        with torch.no_grad():
            train_predictions_full = model(train_x)
            train_r2 = r2_score(
                train_y.cpu().numpy().flatten(), 
                train_predictions_full.cpu().numpy().flatten()
            )

        # Log per epoch metrics with step
        mlflow.log_metrics({
            "train_loss": avg_train_loss,
            "train_mae": avg_train_mae,
            "train_r2": train_r2,
            "train_bias": avg_train_bias,
            "val_loss": avg_val_loss,
            "val_mae": avg_val_mae,
            "val_r2": val_r2,
            "val_bias": avg_val_bias
        }, step=epoch)

        print(f"Epoch {epoch+1}/{params['epochs']} "
              f"train_loss: {avg_train_loss:.4f} "
              f"train_mae: {avg_train_mae:.4f} "
              f"val_loss: {avg_val_loss:.4f} "
              f"val_mae: {avg_val_mae:.4f} cm")

        # Early stopping (uncomment when ready)
        if params.get("patience"):
            if avg_val_loss < best_val_loss:
                best_val_loss = avg_val_loss
                epochs_no_improve = 0
                mlflow.pytorch.log_model(model, artifact_path="best_model")
            else:
                epochs_no_improve += 1

            if epochs_no_improve >= params["patience"]:
                print(f"Early stopping at epoch {epoch+1}")
                break

    # Test evaluation with batching
    model.eval()
    test_losses = []
    test_maes = []
    test_biases = []
    all_test_preds = []
    all_test_true = []
    
    with torch.no_grad():
        for batch_x, batch_y in test_loader:
            test_predictions = model(batch_x)
            loss = loss_fn(test_predictions, batch_y)
            
            test_losses.append(loss.item())
            test_maes.append(torch.mean(torch.abs(test_predictions - batch_y)).item())
            test_biases.append(torch.mean(test_predictions - batch_y).item())
            
            all_test_preds.append(test_predictions.cpu().numpy().flatten())
            all_test_true.append(batch_y.cpu().numpy().flatten())
    
    # Average test metrics
    avg_test_loss = np.mean(test_losses)
    avg_test_mae = np.mean(test_maes)
    avg_test_bias = np.mean(test_biases)
    
    # Calculate test R²
    test_preds_flat = np.concatenate(all_test_preds)
    test_true_flat = np.concatenate(all_test_true)
    test_r2 = r2_score(test_true_flat, test_preds_flat)

    mlflow.log_metrics({
        "test_loss": avg_test_loss,
        "test_mae": avg_test_mae,
        "test_r2": test_r2,
        "test_bias": avg_test_bias,  
    })

    mlflow.pytorch.log_model(model, artifact_path=f"final_model_{params['run_name']}")
    
    print(f"\n✅ Test Results - Loss: {avg_test_loss:.4f}, MAE: {avg_test_mae:.4f}, R²: {test_r2:.4f}")

2026/04/13 17:10:47 INFO mlflow.system_metrics.system_metrics_monitor: Skip logging GPU metrics. Set logger level to DEBUG for more details.
2026/04/13 17:10:47 INFO mlflow.system_metrics.system_metrics_monitor: Started monitoring system metrics.


Epoch 1/1000 train_loss: 0.0141 train_mae: 0.0832 val_loss: 0.0074 val_mae: 0.0593 cm
Epoch 2/1000 train_loss: 0.0088 train_mae: 0.0697 val_loss: 0.0057 val_mae: 0.0545 cm
Epoch 3/1000 train_loss: 0.0076 train_mae: 0.0614 val_loss: 0.0048 val_mae: 0.0427 cm
Epoch 4/1000 train_loss: 0.0061 train_mae: 0.0535 val_loss: 0.0041 val_mae: 0.0407 cm
Epoch 5/1000 train_loss: 0.0055 train_mae: 0.0505 val_loss: 0.0035 val_mae: 0.0375 cm
Epoch 6/1000 train_loss: 0.0049 train_mae: 0.0468 val_loss: 0.0032 val_mae: 0.0341 cm
Epoch 7/1000 train_loss: 0.0043 train_mae: 0.0441 val_loss: 0.0031 val_mae: 0.0335 cm
Epoch 8/1000 train_loss: 0.0044 train_mae: 0.0442 val_loss: 0.0029 val_mae: 0.0313 cm
Epoch 9/1000 train_loss: 0.0044 train_mae: 0.0427 val_loss: 0.0027 val_mae: 0.0302 cm
Epoch 10/1000 train_loss: 0.0045 train_mae: 0.0434 val_loss: 0.0028 val_mae: 0.0313 cm
Epoch 11/1000 train_loss: 0.0042 train_mae: 0.0423 val_loss: 0.0028 val_mae: 0.0301 cm
Epoch 12/1000 train_loss: 0.0039 train_mae: 0.0409 v

2026/04/13 17:39:48 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Epoch 1000/1000 train_loss: 0.0006 train_mae: 0.0136 val_loss: 0.0013 val_mae: 0.0166 cm


2026/04/13 17:39:49 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.
2026/04/13 17:39:51 INFO mlflow.system_metrics.system_metrics_monitor: Stopping system metrics monitoring...
2026/04/13 17:39:51 INFO mlflow.system_metrics.system_metrics_monitor: Successfully terminated system metrics monitoring!



✅ Test Results - Loss: 0.0022, MAE: 0.0227, R²: 0.8326
🏃 View run first_timeseries_with_gru at: http://127.0.0.1:5000/#/experiments/1/runs/5e877d2f12f74a8382b0359068336330
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
